In [1]:
!pip install -q transformers datasets evaluate accelerate torch scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


2025-10-26 15:29:47.712474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761492587.884040      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761492587.936145      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Replace this with your actual file path
file_path = "/kaggle/input/combined-data/Combined Data.csv"

df = pd.read_csv(file_path)
print("Columns:", df.columns.tolist())
df.head()


Columns: ['statement', 'status']


,statement,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety


In [4]:
df = df[['statement', 'status']].dropna()
df.rename(columns={'statement': 'text', 'status': 'label'}, inplace=True)

df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(str).str.strip()

le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

print("Classes:", le.classes_)
df.head()


Classes: ['Anxiety' 'Bipolar' 'Depression' 'Normal' 'Personality disorder' 'Stress'
 'Suicidal']


,text,label,label_id
0,oh my gosh,Anxiety,0
1,"trouble sleeping, confused mind, restless hear...",Anxiety,0
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety,0
3,I've shifted my focus to something else but I'...,Anxiety,0
4,"I'm restless and restless, it's been a month n...",Anxiety,0


In [5]:
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label_id'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['label_id'], random_state=42)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})


In [6]:
model_name = "microsoft/deberta-v3-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    # Create an integer 'label' column
    examples['label'] = examples['label_id']
    # Remove the original label_id column
    del examples['label_id']
    # Tokenize the text with an explicit max_length
    return tokenizer(examples["text"], truncation=True, max_length=256) # Added max_length

encoded = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/38061 [00:00<?, ? examples/s]

Map:   0%|          | 0/6717 [00:00<?, ? examples/s]

Map:   0%|          | 0/7902 [00:00<?, ? examples/s]

In [7]:
num_labels = len(le.classes_)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)


pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
metric = evaluate.load("f1")
recall_metric = evaluate.load("recall")
precision_metric = evaluate.load("precision")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    results = {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": metric.compute(predictions=preds, references=labels, average="macro")["f1"],
        "recall": recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"],
        "precision": precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"],
    }
    return results


In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="recall",   # prioritize recall
    greater_is_better=True,
    per_device_train_batch_size=1, # Reduced batch size further
    per_device_eval_batch_size=1,  # Reduced batch size further
    num_train_epochs=3,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    fp16=True,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    report_to="none",  # Disable Weights & Biases logging
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


/tmp/ipykernel_37/3074832457.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Recall,Precision
1,0.840500,0.878265,0.784279,0.723596,0.755652,0.720473
2,0.683000,0.837111,0.809290,0.772392,0.764115,0.802953
3,0.639500,0.760292,0.847998,0.820749,0.817954,0.824217


TrainOutput(global_step=57093, training_loss=0.7816716654628545, metrics={'train_runtime': 21703.306, 'train_samples_per_second': 5.261, 'train_steps_per_second': 2.631, 'total_flos': 2.1340877328057064e+16, 'train_loss': 0.7816716654628545, 'epoch': 3.0})

In [11]:
preds_output = trainer.predict(encoded["test"])
preds = np.argmax(preds_output.predictions, axis=-1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(test_df["label_id"], preds, target_names=le.classes_))

cm = confusion_matrix(test_df["label_id"], preds)
pd.DataFrame(cm, index=le.classes_, columns=le.classes_)


                      precision    recall  f1-score   support

             Anxiety       0.90      0.88      0.89       576
             Bipolar       0.84      0.87      0.86       417
          Depression       0.82      0.77      0.79      2311
              Normal       0.96      0.96      0.96      2451
Personality disorder       0.78      0.75      0.76       161
              Stress       0.76      0.78      0.77       388
            Suicidal       0.72      0.78      0.75      1598

            accuracy                           0.85      7902
           macro avg       0.83      0.83      0.83      7902
        weighted avg       0.85      0.85      0.85      7902



,Anxiety,Bipolar,Depression,Normal,Personality disorder,Stress,Suicidal
Anxiety,507,21,3,8,14,22,1
Bipolar,10,364,19,4,6,11,3
Depression,7,23,1776,29,7,10,459
Normal,13,6,14,2355,0,50,13
Personality disorder,9,8,14,6,121,1,2
Stress,20,7,12,36,8,304,1
Suicidal,0,2,327,15,0,1,1253


In [12]:
model.save_pretrained("./mental_health_model")
tokenizer.save_pretrained("./mental_health_model")

import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ Model and tokenizer saved successfully!")


✅ Model and tokenizer saved successfully!


In [18]:
from kaggle.api.kaggle_api_extended import KaggleApi
import os
import shutil
import json

api = KaggleApi()
api.authenticate()

# ✅ Step 1: Prepare folder
output_dir = "/kaggle/working/model_upload"
os.makedirs(output_dir, exist_ok=True)

# ✅ Step 2: Copy your file into that folder
shutil.copy("/kaggle/working/mental_health_model/model.safetensors", output_dir)

# ✅ Step 3: Create required metadata file
metadata = {
    "title": "Mental Health Model Upload",
    "id": "abhiindia05/mental-health-model-upload",   # <-- replace with your Kaggle username
    "licenses": [{"name": "CC0-1.0"}],
    "description": "Private upload of a large .safetensors model trained for mental health text classification."
}

with open(os.path.join(output_dir, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f)

# ✅ Step 4: Upload to Kaggle
api.dataset_create_new(
    folder=output_dir,
    dir_mode='zip',
    public=False,
    convert_to_csv=False
)


Starting upload for file model.safetensors


100%|██████████| 1.62G/1.62G [00:19<00:00, 88.3MB/s]


Upload successful: model.safetensors (2GB)


{"ref": "/datasets/abhiindia05/mental-health-model-upload", "url": "https://www.kaggle.com/datasets/abhiindia05/mental-health-model-upload", "status": "Ok", "error": "", "invalidTags": []}

In [16]:
import os
os.environ['KAGGLE_USERNAME'] = "abhiindia05"
os.environ['KAGGLE_KEY'] = "cff16196f16c9022141bef2877f93ea2"
